# **07_Model_Evaluation.ipynb**

Objective:
To assess Logistic Regression and Random Forest models, consider classification metrics, confusion matrices, ROC curves, feature importance, and leakage checks.

Inputs:
    models/lr_elliptic.joblib
    models/lr_ethereum.joblib
    models/scaler_elliptic.joblib
    models/scaler_ethereum.joblib
    models/rf_elliptic.joblib
    models/rf_ethereum.joblib
    models/lr_results.json
    models/rf_results.json
    train_test_data/elliptic_train.csv
    train_test_data/elliptic_test.csv
    train_test_data/ethereum_train.csv
    train_test_data/ethereum_test.csv








Outputs:
    eda_charts/H_elliptic_lr_vs_rf.png
    eda_charts/H_ethereum_lr_vs_rf.png
    eda_charts/I_rf_confusion_matrices.png
    eda_charts/M_roc_curves.png
    eda_charts/J_elliptic_rf_importance.png
    eda_charts/K_ethereum_rf_importance.png

In [3]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

In [10]:
NAVY, TEAL = "#1E3A5F", "#0EA5A6"
base_path = ".."

print("Folder exists:", os.path.exists(base_path))
print("Contents:")

if os.path.exists(base_path):
    print(os.listdir(base_path))

Folder exists: True
Contents:
['cleaned_data', 'train_test_data', 'eda_charts', '.DS_Store', 'Raw_data', 'models', 'results', 'notebooks']


In [11]:
print(os.listdir("../models"))

['scaler_elliptic.joblib', 'rf_results.json', 'scaler_ethereum.joblib', 'lr_results.json', 'lr_ethereum.joblib', 'rf_ethereum.joblib', 'rf_elliptic.joblib', 'lr_elliptic.joblib']


In [12]:
with open("../models/lr_results.json") as f:
    lr_results = json.load(f)
with open("../models/rf_results.json") as f:
    rf_results = json.load(f)

In [13]:
lr_ell = joblib.load("../models/lr_elliptic.joblib")
scaler_ell = joblib.load("../models/scaler_elliptic.joblib")
rf_ell = joblib.load("../models/rf_elliptic.joblib")
lr_eth = joblib.load("../models/lr_ethereum.joblib")
scaler_eth = joblib.load("../models/scaler_ethereum.joblib")
rf_eth = joblib.load("../models/rf_ethereum.joblib")

In [14]:
ell_train, ell_test = pd.read_csv("../train_test_data/elliptic_train.csv"), pd.read_csv("../train_test_data/elliptic_test.csv")
eth_train, eth_test = pd.read_csv("../train_test_data/ethereum_train.csv"), pd.read_csv("../train_test_data/ethereum_test.csv")
feat_cols_ell = [c for c in ell_train.columns if c.startswith("feat_")]
feat_cols_eth = [c for c in eth_train.columns if c not in ["Address", "FLAG"]]

In [17]:
import os

EDA_PATH = "../eda_charts"
os.makedirs(EDA_PATH, exist_ok=True)

print("Charts will be saved to:", os.path.abspath(EDA_PATH))

Charts will be saved to: /Users/dhilnasherin/Downloads/Dissertation_dilna/ML_Pipeline/eda_charts


In [19]:
# Metrics comparison bar charts
metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]
for dataset, lr_key, rf_key in [("elliptic", "elliptic_lr", "elliptic_rf"), ("ethereum", "ethereum_lr", "ethereum_rf")]:
    lr_vals = [lr_results[lr_key][m] for m in metrics]
    rf_vals = [rf_results[rf_key][m] for m in metrics]
    x = np.arange(len(metrics))
    fig, ax = plt.subplots(figsize=(8, 4.5), dpi=150)
    ax.bar(x - 0.18, lr_vals, width=0.36, label="Logistic Regression", color=TEAL)
    ax.bar(x + 0.18, rf_vals, width=0.36, label="Random Forest", color=NAVY)
    ax.set_xticks(x); ax.set_xticklabels(["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"])
    ax.set_title(f"{dataset.title()}: Logistic Regression vs Random Forest")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(EDA_PATH, f"H_{dataset}_lr_vs_rf.png"))
    plt.close(fig)
    print(f"{dataset}: LR F1={lr_results[lr_key]['f1']:.3f}, RF F1={rf_results[rf_key]['f1']:.3f}")

elliptic: LR F1=0.303, RF F1=0.752
ethereum: LR F1=0.516, RF F1=0.951


In [21]:
# Confusion matrices (Random Forest)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), dpi=150)
for ax, (key, title) in zip(axes, [("elliptic_rf", "Elliptic"), ("ethereum_rf", "Ethereum")]):
    cm = np.array(rf_results[key]["confusion_matrix"])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max()/2 else "black")
    ax.set_title(f"{title} \u2014 Random Forest")
fig.tight_layout()
fig.savefig("../eda_charts/I_rf_confusion_matrices.png")
plt.close(fig)

In [22]:
# ROC curves
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), dpi=150)
for ax, (X_ell_or_eth, y, lr_m, scaler, rf_m, title) in zip(axes, [
    (ell_test[feat_cols_ell], ell_test["label"], lr_ell, scaler_ell, rf_ell, "Elliptic"),
    (eth_test[feat_cols_eth], eth_test["FLAG"], lr_eth, scaler_eth, rf_eth, "Ethereum"),
]):
    prob_lr = lr_m.predict_proba(scaler.transform(X_ell_or_eth))[:, 1]
    prob_rf = rf_m.predict_proba(X_ell_or_eth)[:, 1]
    fpr_lr, tpr_lr, _ = roc_curve(y, prob_lr)
    fpr_rf, tpr_rf, _ = roc_curve(y, prob_rf)
    ax.plot(fpr_lr, tpr_lr, color=TEAL, label=f"LR (AUC={roc_auc_score(y, prob_lr):.3f})")
    ax.plot(fpr_rf, tpr_rf, color=NAVY, label=f"RF (AUC={roc_auc_score(y, prob_rf):.3f})")
    ax.plot([0, 1], [0, 1], color="gray", linestyle="--")
    ax.set_title(f"{title}: ROC Curve"); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("../eda_charts/M_roc_curves.png")
plt.close(fig)

In [25]:
# Feature importance (RQ1)
imp_ell = pd.Series(rf_ell.feature_importances_, index=feat_cols_ell).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(8, 5.5), dpi=150)
ax.barh(imp_ell.index[::-1], imp_ell.values[::-1], color=NAVY)
ax.set_title("Elliptic: Top 15 Feature Importances (RF)")
fig.tight_layout()
fig.savefig(os.path.join(EDA_PATH, "J_elliptic_rf_importance.png"))
plt.close(fig)

imp_eth = pd.Series(rf_eth.feature_importances_, index=feat_cols_eth).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(8, 5.5), dpi=150)
ax.barh(imp_eth.index[::-1], imp_eth.values[::-1], color=NAVY)
ax.set_title("Ethereum: Top 15 Feature Importances (RF)")
fig.tight_layout()
fig.savefig(os.path.join(EDA_PATH, "K_ethereum_rf_importance.png"))
plt.close(fig)

In [26]:
print(f"\nTop Elliptic feature: {imp_ell.index[0]} ({imp_ell.values[0]:.4f})")
print(f"Top Ethereum feature: {imp_eth.index[0]} ({imp_eth.values[0]:.4f})")


Top Elliptic feature: feat_48 (0.0524)
Top Ethereum feature: has_erc20_activity (0.1302)


In [27]:
# Leakage / robustness check
overlap = set(eth_train["Address"]) & set(eth_test["Address"])
print(f"\nLeakage check -- Ethereum train/test address overlap: {len(overlap)} (should be 0)")



Leakage check -- Ethereum train/test address overlap: 0 (should be 0)


In [28]:
eth_full = pd.concat([eth_train, eth_test])
suspects = []
for c in feat_cols_eth:
    try:
        auc = roc_auc_score(eth_full["FLAG"], eth_full[c].fillna(0))
        auc = max(auc, 1 - auc)
        if auc > 0.85:
            suspects.append((c, round(auc, 3)))
    except Exception:
        pass
print(f"Leakage check -- single-feature AUC > 0.85: {suspects or 'none found'}")


Leakage check -- single-feature AUC > 0.85: none found


In [29]:
print("\nAll evaluation outputs saved to eda_charts/")


All evaluation outputs saved to eda_charts/


In [34]:
print("Evaluation charts saved successfully.")
print("Location:", os.path.abspath("../eda_charts"))

Evaluation charts saved successfully.
Location: /Users/dhilnasherin/Downloads/Dissertation_dilna/ML_Pipeline/eda_charts
